# Modern LangChain 1.x RAG Tutorial
This notebook demonstrates a simple Retrieval-Augmented Generation (RAG) pipeline using LangChain 1.x, Chroma, and OpenAI.

In [4]:
# Install (run once)
# !pip install langchain langchain-openai langchain-chroma python-dotenv


In [5]:
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()


True

In [6]:
documents = [
    Document(page_content="Machine Learning is a branch of Artificial Intelligence that enables computers to learn from data."),
    Document(page_content="Deep Learning is a subset of Machine Learning that uses deep neural networks."),
    Document(page_content="CNN stands for Convolutional Neural Network. CNNs are mainly used for image classification and computer vision."),
    Document(page_content="RNN stands for Recurrent Neural Network. RNNs are used for sequential data.")
]
documents

[Document(metadata={}, page_content='Machine Learning is a branch of Artificial Intelligence that enables computers to learn from data.'),
 Document(metadata={}, page_content='Deep Learning is a subset of Machine Learning that uses deep neural networks.'),
 Document(metadata={}, page_content='CNN stands for Convolutional Neural Network. CNNs are mainly used for image classification and computer vision.'),
 Document(metadata={}, page_content='RNN stands for Recurrent Neural Network. RNNs are used for sequential data.')]

In [7]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="demo_collection"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store created successfully.")


Vector store created successfully.


In [8]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

system_prompt = '''
You are an AI assistant.

Use ONLY the provided context to answer.

If the answer is not present in the context,
reply with "I don't know."

Context:
{context}
'''

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])


In [9]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain)


first={
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000276C73EEA90>, search_kwargs={'k': 2})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nYou are an AI assistant.\n\nUse ONLY the provided context to answer.\n\nIf the answer is not present in the context,\nreply with "I don\'t know."\n\nContext:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version

In [10]:
question = "What is Deep Learning?"

print("Question:", question)
print()

docs = retriever.invoke(question)

print("Retrieved Documents:")
for i, doc in enumerate(docs, start=1):
    print(f"\nDocument {i}")
    print(doc.page_content)

print("\nAnswer:")
print(rag_chain.invoke(question))


Question: What is Deep Learning?

Retrieved Documents:

Document 1
Deep Learning is a subset of Machine Learning that uses deep neural networks.

Document 2
Deep Learning is a subset of Machine Learning that uses deep neural networks.

Answer:
Deep Learning is a subset of Machine Learning that uses deep neural networks.
